In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_features=10, 
    n_samples=1000, 
    n_informative=8,
    n_redundant=2,
    n_repeated=0,
    n_classes=2, 
    random_state=42
)

### Method 1: Evaluate the model using train, test split and tune parameters by trial and error

In [2]:
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = DecisionTreeClassifier(criterion="entropy", max_depth=10) # criteria: "gini" or "entropy", max_depth=5 or 10
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.84      0.74      0.79       130
           1       0.75      0.85      0.80       120

    accuracy                           0.79       250
   macro avg       0.80      0.79      0.79       250
weighted avg       0.80      0.79      0.79       250



### Method 2: Use K Fold Cross Validation

In [4]:
from sklearn.model_selection import cross_val_score

cross_val_score(DecisionTreeClassifier(criterion="gini", max_depth=5), X, y, cv=5)

array([0.775, 0.79 , 0.745, 0.79 , 0.77 ])

In [5]:
cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=5), X, y, cv=5)

array([0.765, 0.785, 0.75 , 0.815, 0.775])

In [6]:
criterion = ["gini", "entropy"]
max_depth = [5, 10, 15]

avg_scores = {}

for c in criterion:
    for d in max_depth:
        clf = DecisionTreeClassifier(criterion=c, max_depth=d)
        score_list = cross_val_score(clf, X, y, cv=5)
        avg_scores[c + "_" + str(d)] = np.average(score_list)
        
avg_scores

{'gini_5': np.float64(0.784),
 'gini_10': np.float64(0.788),
 'gini_15': np.float64(0.8),
 'entropy_5': np.float64(0.7809999999999999),
 'entropy_10': np.float64(0.793),
 'entropy_15': np.float64(0.8089999999999999)}

In [7]:
from sklearn.model_selection import GridSearchCV

clf=GridSearchCV(
    DecisionTreeClassifier(),
    {
        'criterion' : ['gini','entropy'], 
        'max_depth' : [5,10,15]
    },
    cv=5,
    return_train_score=False #if true it will evaluate on train data as well
)
clf.fit(X,y)
clf.cv_results_

{'mean_fit_time': array([0.01460829, 0.02712145, 0.01778231, 0.0148798 , 0.02266779,
        0.02961555]),
 'std_fit_time': array([0.00187434, 0.00968928, 0.00086779, 0.00034029, 0.00241357,
        0.00126314]),
 'mean_score_time': array([0.00210509, 0.00263448, 0.00133104, 0.00141459, 0.0015337 ,
        0.00183973]),
 'std_score_time': array([6.64648408e-04, 1.47731898e-03, 1.33745419e-04, 1.36523350e-04,
        2.69578756e-04, 6.19584890e-05]),
 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'entropy', 'entropy',
                    'entropy'],
              mask=[False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'param_max_depth': masked_array(data=[5, 10, 15, 5, 10, 15],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'params': [{'criterion': 'gini', 'max_depth': 5},
  {'criterion': 'gini', 'max_depth': 10},
  {'criterion': 'gini', 'max_depth': 15},
  {'criterion': '

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)

print(clf.best_params_)
print(clf.best_score_)  # CV score from training data

test_score = clf.score(X_test, y_test)
print("Test accuracy:", test_score)

{'criterion': 'entropy', 'max_depth': 10}
0.80375
Test accuracy: 0.8


In [8]:
df=pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.014608,0.001874,0.002105,0.000665,gini,5,"{'criterion': 'gini', 'max_depth': 5}",0.775,0.810,0.760,0.795,0.770,0.782,0.018055,5
1,0.027121,0.009689,0.002634,0.001477,gini,10,"{'criterion': 'gini', 'max_depth': 10}",0.775,0.750,0.785,0.785,0.815,0.782,0.020881,4
2,0.017782,0.000868,0.001331,0.000134,gini,15,"{'criterion': 'gini', 'max_depth': 15}",0.810,0.745,0.815,0.825,0.835,0.806,0.031686,2
3,0.014880,0.000340,0.001415,0.000137,entropy,5,"{'criterion': 'entropy', 'max_depth': 5}",0.765,0.780,0.755,0.815,0.780,0.779,0.020347,6
4,0.022668,0.002414,0.001534,0.000270,entropy,10,"{'criterion': 'entropy', 'max_depth': 10}",0.770,0.780,0.830,0.770,0.795,0.789,0.022450,3
5,0.029616,0.001263,0.001840,0.000062,entropy,15,"{'criterion': 'entropy', 'max_depth': 15}",0.775,0.800,0.850,0.780,0.855,0.812,0.034147,1


In [10]:
df=df[['param_criterion','param_max_depth','mean_test_score']]
df

,param_criterion,param_max_depth,mean_test_score
0,gini,5,0.782
1,gini,10,0.782
2,gini,15,0.806
3,entropy,5,0.779
4,entropy,10,0.789
5,entropy,15,0.812


In [12]:
clf.best_params_

{'criterion': 'entropy', 'max_depth': 10}

In [13]:
#we already got the best parameter
#now if we want the model associated with it
clf.best_estimator_
#now we caan take this estimator export it to a file joblib or what ever file and then deploy that to production

,criterion,'entropy'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


<img src="1.png" alt="Explaination" >

<img src="2.png" >

In [ ]:
#now we want to try with diff models and params 
from sklearn.svm import SVC 
